# Quickstart — backtest your first bot in 30 seconds

This notebook walks through:

1. **Importing the SDK** (`pip install -e .` from the repo root, or `pip install cvz-backtester` once published).
2. **Building or downloading candles**. We default to a synthetic sine-wave dataset so the notebook is fully offline; toggle a flag to fetch real BTCUSDT data from Binance.
3. **Running an EMA-crossover backtest** and printing the summary metrics.
4. **Plotting the equity curve** so you can eyeball the strategy.

In [ ]:
import math
from decimal import Decimal

import matplotlib.pyplot as plt

from backtester import (
    BacktestConfig,
    BacktestEngine,
    Candle,
    EMACross,
    __version__,
)

print(f"cvz-backtester v{__version__}")

## 1. Build a candle dataset

Two options: synthetic (always works, no network) or real Binance data (requires `examples/notebooks/.duckdb` from a prior CLI download).

In [ ]:
USE_BINANCE = False  # flip to True if you've already downloaded candles via the CLI

def synthetic_candles(n: int = 720) -> list[Candle]:
    """30 days of fake 1h candles with a sine pattern + drift so EMA crosses fire."""
    base = 100.0
    out = []
    for i in range(n):
        offset = 25 * math.sin(i / 20.0) + i * 0.02
        close = base + offset
        open_ = base + 25 * math.sin((i - 1) / 20.0) + (i - 1) * 0.02 if i > 0 else close
        high = max(open_, close) + 0.5
        low = min(open_, close) - 0.5
        out.append(
            Candle(
                timestamp_ms=i * 3_600_000,
                open=Decimal(str(open_)),
                high=Decimal(str(high)),
                low=Decimal(str(low)),
                close=Decimal(str(close)),
                volume=Decimal("100"),
            )
        )
    return out

if USE_BINANCE:
    from pathlib import Path
    from backtester import BinanceDownloader
    db_path = Path(".") / "candles.duckdb"
    downloader = BinanceDownloader(db_path)
    rows = downloader.load_candles("BTCUSDT", "1h")
    candles = [Candle.from_dict(r) for r in rows]
    if not candles:
        print("No candles in DB — falling back to synthetic data.")
        candles = synthetic_candles()
else:
    candles = synthetic_candles()

print(f"Loaded {len(candles)} candles, first close = {float(candles[0].close):.2f}")

## 2. Run the backtest

We use the stock `EMACross` bot. Its `param_spec()` exposes everything tweakable from the UI — here we just hard-code the values.

In [ ]:
bot = EMACross(fast_ema=12, slow_ema=26, stop_loss_pct=0.05, profit_factor=0.05)
engine = BacktestEngine(
    BacktestConfig(
        initial_cash=Decimal("10000"),
        taker_fee_pct=Decimal("0.1"),
        slippage_pct=Decimal("0.05"),
    )
)
result = engine.run(bot, candles, symbol="BTCUSDT", timeframe="1h", bot_names=["EMACross"])
print(result.summary())

## 3. Plot equity + drawdown

Drawdown is computed against the rolling peak — a quick sanity check on whether the strategy survives its worst stretch.

In [ ]:
equity = [float(e) for e in result.equity_curve]
peak = []
running = equity[0]
for v in equity:
    if v > running:
        running = v
    peak.append(running)
drawdown_pct = [(p - v) / p * 100 if p > 0 else 0 for v, p in zip(equity, peak)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(equity, color="tab:blue", linewidth=1.5)
ax1.set_ylabel("Equity (USDT)")
ax1.set_title("EMACross on synthetic candles")
ax1.grid(alpha=0.3)
ax2.fill_between(range(len(drawdown_pct)), drawdown_pct, color="tab:red", alpha=0.4)
ax2.set_ylabel("Drawdown %")
ax2.set_xlabel("Candle index")
ax2.invert_yaxis()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## What now?

- Tweak the bot params in cell 4 and re-run the cell to compare.
- Open [`02_compare_bots.ipynb`](02_compare_bots.ipynb) to run all five stock bots side-by-side on the same dataset.
- Try the **Strategy DSL** (`from backtester import DSLBot`) to write a bot without Python — see `backtester/README.md`.